In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.optimize import curve_fit

from myfit import *
from myfunctions import *
from mysetup import *

from jup_plot import *

def gauss (x, a, lista, method):
    
    # a is the dictionary
    # lista is the list of codes to consider
    # method is the algorithm, eg 'LA'
    import scipy.stats as stats
    gauss = 0
    for i in lista:
        gauss += stats.norm.pdf(x, a[i][method][0], a[i][method][1])
    gauss /= len(lista)
    return gauss

def mean_codes(a,lista,method):
    energy = 0
    norm = 0
    error = 0
    
    for i in lista:
        energy += a[i][method][0] * (1/a[i][method][1]**2)
        norm += (1/a[i][method][1]**2)
    
    energy /= norm
    #error = np.sqrt(1/norm)
    
    for i in lista:
        error += (a[i][method][0]-energy)**2 / (len(lista)-1)
    
    error = np.sqrt(error)
    
    return energy, error


def gauss_tau (x, a, lista, tau):
    
    # a is the dictionary
    # lista is the list of codes to consider
    # method is the algorithm, eg 'LA'
    import scipy.stats as stats
    gauss = 0
    for i in lista:
        gauss += stats.norm.pdf(x, a[i].loc[tau][0], a[i].loc[tau][1])
    gauss /= len(lista)
    return gauss

In [4]:
codes_all = ['TurboRVB', 'TurboRVB_DMCv1', 'TurboRVB_DMCv2', 'CASINO', 'CASINO_limdmc5', 'CASINO_limdmc6t', 'QMCPACK', 'CHAMP_Holland', 'CHAMP_Cornell_Paris', 'QMC=Chem', 'QMeCha', 'PyQMC', 'Amolqc', 'QWalk', 'CMQMC', 'TurboRVB_sandro', 'QMCPACK_complex', 'CASINO_limdmc4', 'CASINO_limdmc5_j+', 'CASINO_limdmc5_cbycJbufF', 'CASINO_limdmc5t']

list_tau=['CASINO','Amolqc','CMQMC','PyQMC','QMC=Chem','QMCPACK_final']


codes_opt = ['TurboRVB_sandro', ]

code_final_list = ['Amolqc', 'CASINO', 'CHAMP_Cornell_Paris', 'CHAMP_Holland', 'CMQMC', 'PyQMC', 'QMC=Chem', 'QMCPACK_final', 'QMeCha', 'QWalk', 'TurboRVB', 'TurboRVB_DMCv2']

systems = [ 'm', 'w', 'mw', 'm---w' ]
methods = [ 'LA', 'TM', 'DLA', 'DLTM' ]

colors = {
    'Amolqc': '#e6194B',
    'CASINO': '#f58231',
    'CHAMP_Cornell_Paris':'#ffe119',
    'CHAMP_Holland': '#bfef45',
    'CMQMC': '#3cb44b',
    'PyQMC': '#42d4f4',
    'QMC=Chem': '#4363d8',
    'QMCPACK_final': '#911eb4',
    'QMeCha': '#f032e6',
    'QWalk': '#800000',
    'TurboRVB': '#808000',
    'TurboRVB_DMCv2': '#469990'
}

taumax = 0.1

def x_axis( df, code ):
    x = np.array( df.index )
    if code.startswith('TurboRVB') and not ('DMC' in code):
        xnew = x**2
    elif code.startswith('QMC=Chem'):
        xnew = x #* 20
    else:
        xnew = x
    return xnew

In [11]:
data = {}
dataEb = {}
dataEb_fit = {} # Fitted binding energy and errors
final_Eb = {} # Final extrapolated binding energy to timestep = 0
Eb_DLA = {}
for code in codes_all:
    data[code] = {}
    dataEb[code] = {}
    dataEb_fit[code] = {}

for code in code_final_list:
    final_Eb[code] = {}
    Eb_DLA[code] = {}

for code in codes_all:
    if code.startswith('TurboRVB') and not ('DMC' in code):
        names = ['a','ene','err']
    else:
        names = ['tau','ene','err']
    for method in methods+methods_opt:
        if 'QMCPACK' in code and method == 'DLTM':
            continue
        for syst in systems:
            key = f'{syst}_{method}'
            try: 
                data[code][key] = pd.read_csv(
                    f'../{code}/energy_{key}.csv',
                    names=names, decimal='.',
                    header=None, index_col=names[0] )
            except:
                data[code][key] = None
                #print(f'No data for {code}, {method}, {syst}')

# Compute binding energies
# Hartree to meV
c = 27.211396641308*1000

for code in codes_all:
    for method in methods:
        try:
            # Computing Binding energy for bound - far
            mw = data[code][f'mw_{method}']
            far = data[code][f'm---w_{method}']
            Eb = c*(mw.ene - far.ene)
            Eb_err = c*(far.err**2+mw.err**2)**0.5
            df_Eb = pd.DataFrame([Eb, Eb_err]).T
            dataEb[code][f'Eb_far_{method}'] = df_Eb.dropna()

            # Calculate the size consistency error
            m = data[code][f'm_{method}']
            w = data[code][f'w_{method}']
            far = data[code][f'm---w_{method}']
            Eb = c*(far.ene-m.ene-w.ene)
            Eb_err = c*(far.err**2+m.err**2+w.err**2)**0.5
            df_Eb = pd.DataFrame([Eb, Eb_err]).T
            dataEb[code][f'Eb_SCE_{method}'] = df_Eb.dropna()

            # Computing Binding energy for bound - methane - water

            m = data[code][f'm_{method}']
            w = data[code][f'w_{method}']
            mw = data[code][f'mw_{method}']
            Eb = c*(mw.ene-m.ene-w.ene)
            Eb_err = c*(mw.err**2+m.err**2+w.err**2)**0.5
            df_Eb = pd.DataFrame([Eb, Eb_err]).T
            dataEb[code][f'Eb_{method}'] = df_Eb.dropna()
            if method == 'DLA':
                if code in code_final_list:
                    Eb_DLA[code] = df_Eb.dropna()
                elif code == 'CASINO_limdmc5':
                    Eb_DLA['CASINO'] = df_Eb.dropna()
                elif code == 'QMCPACK' and method in ['TM','DLA']:
                    Eb_DLA['QMCPACK_final'] = df_Eb.dropna()
                elif code == 'QMCPACK_complex' and method == 'LA':
                    Eb_DLA['QMCPACK_final'] = df_Eb.dropna()


            # Fit curve to the data
            datac1 = df_Eb
            datac1['x'] = x_axis( datac1, code )
            datac = datac1 #[ datac1.x <= taumax ]
            datac = datac.dropna()
            x = datac.x
            y = datac.ene
            y_err = datac.err
            fun = fun_quad
            xdata = x.to_numpy()
            ydata = y.to_numpy()
            sigma = y_err.to_numpy()
            popt, pcov = curve_fit( fun, xdata=xdata, ydata=ydata, sigma=sigma )
            chi2, ndof, alpha = get_chi2_alpha_fitfun( xdata, ydata, sigma, fun, popt )
            print(code+'_'+method,chi2/ndof,chi2,ndof,alpha)
            
            space_pts = 101
            xlinspace = np.linspace( 0, taumax, space_pts)
            funlinspace = fun( xlinspace, *popt )

            # Fit error curve to the data
            xfit, m, s = fit_err(xdata,ydata,sigma,fitfun=fun,xmax=taumax,Nx=space_pts)


            dataEb_fit[code][f'Eb_{method}'] = {'x': xdata, 'Eb': ydata , 'Eb_err': y_err, 'x_fit': xlinspace, 'Eb_fit': funlinspace ,  'Eb_fit_lower_lim': m-1*s,  'Eb_fit_upper_lim': m+1*s}
            lim_Eb = funlinspace[0]
            lim_Eb_upper = (m+1*s)[0]
            lim_Eb_lower = (m-1*s)[0]
            lim_Eb_sigma = np.max([abs(lim_Eb - lim_Eb_upper), abs(lim_Eb - lim_Eb_lower)])

            if code in code_final_list:
                final_Eb[code][method] = [lim_Eb, lim_Eb_sigma]
            elif code == 'CASINO_limdmc5':
                final_Eb['CASINO'][method] = [lim_Eb, lim_Eb_sigma]
            elif code == 'QMCPACK' and method in ['TM','DLA']:
                final_Eb['QMCPACK_final'][method] = [lim_Eb, lim_Eb_sigma]
            elif code == 'QMCPACK_complex' and method == 'LA':
                final_Eb['QMCPACK_final'][method] = [lim_Eb, lim_Eb_sigma]
        except:
            pass
np.save('DMC_DLA_timestep.npy',Eb_DLA, allow_pickle=True)
np.save('final_DMC_numbers.npy',final_Eb,allow_pickle=True)

TurboRVB_TM 0.6848251845155727 2.0544755535467183 3 0.5611774251135817
TurboRVB_DLA 0.6409042067734946 1.9227126203204836 3 0.5886019011591661
TurboRVB_DLTM 0.30250065178884283 0.9075019553665286 3 0.8236170474252489
TurboRVB_DMCv1_TM 2.08186789744587 6.2456036923376095 3 0.10025366058414253
TurboRVB_DMCv2_TM 1.479402564026767 8.876415384160602 6 0.18064860701289243
CASINO_limdmc5_LA 1.149730668503485 2.29946133700697 2 0.3167220611193622
CASINO_limdmc5_TM 0.6705972194096774 1.3411944388193549 2 0.5114030667294371
CASINO_limdmc5_DLA 1.7166815862637004 5.150044758791101 3 0.16113411699763625
CASINO_limdmc5_DLTM 1.2425465057009983 3.727639517102995 3 0.29241580181746496
CASINO_limdmc6t_LA 2.280691723954791 9.122766895819163 4 0.05810308322179247
CASINO_limdmc6t_TM 1.8063425269540567 9.031712634770283 5 0.10780591363877357
CASINO_limdmc6t_DLA 6.234331189959823 24.937324759839292 4 5.179086901530017e-05
CASINO_limdmc6t_DLTM 1.9088998371908865 11.45339902314532 6 0.07533443406911633
QMCPACK

# Figure 1 - Comparison among different codes

In [55]:
# Numbering dictionary
code_to_num_dict = {x: index for index, x in enumerate(code_final_list)}

method_codes_dict = {
    'LA': ['CASINO','QMCPACK_final','CMQMC','CHAMP_Holland','CHAMP_Cornell_Paris','QMeCha','PyQMC','QWalk'],
    'TM': ['CASINO','TurboRVB','TurboRVB_DMCv2','QMCPACK_final','CMQMC','CHAMP_Holland','CHAMP_Cornell_Paris','PyQMC','QWalk'],
    'DLA': ['CASINO','TurboRVB','QMCPACK_final','CMQMC','QMC=Chem','QMeCha','PyQMC','Amolqc'],
    'DLTM': ['CASINO','TurboRVB','CMQMC']
}

# figure 1 Comparison among different codes 
fig , ax = plt.subplots(4,1,figsize=(6.67,8),dpi=600,constrained_layout=True,sharex=True,sharey=True)

counter = 0
for method in ['LA','TM','DLA','DLTM']:
    ax[counter].axhline(y=-27,color='black',ls='--')

    for code in method_codes_dict[method]:
        ax[counter].errorbar(code_to_num_dict[code],final_Eb[code][method][0],final_Eb[code][method][1],fmt='o',capsize=6,ms=10,elinewidth=1.5,markeredgewidth=1,mec='black',c=colors[code])

    ax[counter].axhline(mean_codes(final_Eb,method_codes_dict[method],method)[0],c='grey',ls='dashed',lw=2)
    ax[counter].fill_between(np.arange(-0.5,11.5,0.1),mean_codes(final_Eb,method_codes_dict[method],method)[0] + mean_codes(final_Eb,method_codes_dict[method],method)[1],mean_codes(final_Eb,method_codes_dict[method],method)[0] - mean_codes(final_Eb,method_codes_dict[method],method)[1] ,color='grey',alpha=.3)
    ax[counter].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
    ax[counter].set_xticklabels([])
    ax[counter].set_xlim([-0.5,11.5])
    ax[counter].set_ylim([-55,-15])
    ax[counter].set_ylabel('Binding energy [meV]')

    counter += 1

ax[0].set_title('Locality Approximation (LA)')
ax[1].set_title('T-Move (TM)')
ax[2].set_title('Determinant Locality Approximation (DLA)')
ax[3].set_title('Determinant Locality Approximation with T-Move (DLTM)',fontsize=11)

ax[3].set_xticklabels(['Amolqmc','CASINO','CHAMP-Cornell-Paris','CHAMP-Holland','CMQMC','PyQMC','QMC=Chem','QMCPACK','QMeCha','QWalk','TurboRVB','TurboRVB-DMC-v2'],rotation=90) 

plt.savefig('Figures/Fig_01.png')

# Figure 2 - Distribution of errors for different methods

In [57]:
x=np.arange(-55,-20,0.1)

fig , ax = plt.subplots(figsize=(3.37,3.5),dpi=600,constrained_layout=True)

ax.axvline(-27,ls='dashed',lw=2,c='black')

ax.plot(x,gauss(x,final_Eb,method_codes_dict['LA'],'LA')+0.48*3,label='LA', lw=2,c='blue')
ax.fill_between(x,gauss(x,final_Eb,method_codes_dict['LA'],'LA')+0.48*3,0.48*3,color='blue',alpha=0.2)
ax.axhline(0.48*3,ls='dashed',lw=0.5,c='blue')



ax.plot(x,gauss(x,final_Eb,method_codes_dict['TM'],'TM')+0.48*2,label='TM', lw=2,c='orange')
ax.fill_between(x,gauss(x,final_Eb,method_codes_dict['TM'],'TM')+0.48*2,0.48*2,color='orange',alpha=0.2)
ax.axhline(0.48*2,ls='dashed',lw=0.5,c='orange')



ax.plot(x,gauss(x,final_Eb,method_codes_dict['DLA'],'DLA')+0.48,label='DLA', lw=2,c='red')
ax.fill_between(x,gauss(x,final_Eb,method_codes_dict['DLA'],'DLA')+0.48,0.48,color='red',alpha=0.2)
ax.axhline(0.48,ls='dashed',lw=0.5,c='red')



ax.plot(x,gauss(x,final_Eb,method_codes_dict['DLTM'],'DLTM'),label='DLTM', lw=2,c='green')
ax.fill_between(x,gauss(x,final_Eb,method_codes_dict['DLTM'],'DLTM'),0,color='green',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='green')


plt.yticks([])
plt.xlim([-55,-20])
plt.ylabel('Probability distribution')
plt.xlabel('Binding energy [meV]')
plt.legend(loc='lower left',shadow=False,framealpha=1.,facecolor='white',edgecolor='black')
plt.savefig('Figures/Fig_02.png')


# Figure 3 - Timestep dependence of distribution of codes

In [64]:
x=np.arange(-60,-10,0.1)
offset=0.25

fig , ax = plt.subplots(figsize=(3.37,4),dpi=600)

ax.axvline(-27,ls='dashed',lw=2,c='black')

ax.plot(x,gauss_tau(x,Eb_DLA,list_tau,0.0400)+offset*5,label=r'$\tau$ = 0.04 au',lw=2,c='blue')
ax.fill_between(x,gauss_tau(x,Eb_DLA,list_tau,0.0400)+offset*5,offset*5,color='blue',alpha=0.2)
ax.axhline(offset*5,ls='dashed',lw=0.5,c='blue')


ax.plot(x,gauss_tau(x,Eb_DLA,list_tau,0.0200)+offset*4,label=r'$\tau$ = 0.02 au',lw=2,c='dodgerblue')
ax.fill_between(x,gauss_tau(x,Eb_DLA,list_tau,0.0200)+offset*4,offset*4,color='dodgerblue',alpha=0.2)
ax.axhline(offset*4,ls='dashed',lw=0.5,c='dodgerblue')

ax.plot(x,gauss_tau(x,Eb_DLA,list_tau,0.0100)+offset*3,label=r'$\tau$ = 0.01 au',lw=2,c='turquoise')
ax.fill_between(x,gauss_tau(x,Eb_DLA,list_tau,0.0100)+offset*3,offset*3,color='turquoise',alpha=0.2)
ax.axhline(offset*3,ls='dashed',lw=0.5,c='turquoise')

ax.plot(x,gauss_tau(x,Eb_DLA,list_tau,0.0050)+offset*2,label=r'$\tau$ = 0.05 au',lw=2,c='orange')
ax.fill_between(x,gauss_tau(x,Eb_DLA,list_tau,0.005)+offset*2,offset*2,color='orange',alpha=0.2)
ax.axhline(offset*2,ls='dashed',lw=0.5,c='orange')

ax.plot(x,gauss_tau(x,Eb_DLA,list_tau,0.0025)+offset,label=r'$\tau$ = 0.025 au', lw=2,c='firebrick')
ax.fill_between(x,gauss_tau(x,Eb_DLA,list_tau,0.0025)+offset,offset,color='firebrick',alpha=0.2)
ax.axhline(offset,ls='dashed',lw=0.5,c='firebrick')

ax.plot(x,gauss(x,final_Eb,method_codes_dict['DLA'],'DLA'),label=r'$\tau$ $\to$ 0 au', lw=2,c='red')
ax.fill_between(x,gauss(x,final_Eb,method_codes_dict['DLA'],'DLA'),0,color='red',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='red')


plt.yticks([])
plt.xlim([-60,-15])
plt.ylabel('Probability distribution')
plt.xlabel('Binding energy [meV]')
plt.legend(loc='lower left',shadow=False,framealpha=1.,facecolor='white',edgecolor='black')

plt.tight_layout()
plt.savefig('Figures/Fig_03.png')



# Figure S1 - Time-step dependence of binding energy of individual codes

In [66]:
dataEb_fit['CASINO'] = dataEb_fit['CASINO_limdmc5']
dataEb_fit['QMCPACK_final'] = {'Eb_LA': dataEb_fit['QMCPACK_complex']['Eb_LA'],
                               'Eb_TM': dataEb_fit['QMCPACK']['Eb_TM'],
                               'Eb_DLA': dataEb_fit['QMCPACK']['Eb_DLA']}
# Plotting the binding energy curves for all of the codes

fig, axs = plt.subplots(2,4, figsize=(6.69,4.5), sharey=True, sharex=True,dpi=600,constrained_layout=True)

# Plot the top row binding energies
for indexj, j in enumerate(['Eb_LA','Eb_TM','Eb_DLA','Eb_DLTM']):
    for i in ['CASINO', 'TurboRVB','TurboRVB_DMCv2','QMCPACK_final','CMQMC']:
        if i == 'TurboRVB_DMCv2' and j != 'Eb_TM':
            continue
        if j == 'Eb_LA' and i == 'TurboRVB':
            continue
        if j == 'Eb_DLTM' and i == 'QMCPACK_final':
            continue
        code_name = i

        axs[0,indexj].errorbar(dataEb_fit[code_name][j]['x'], dataEb_fit[code_name][j]['Eb'],yerr = dataEb_fit[code_name][j]['Eb_err'], fmt='o', markersize=3,elinewidth=1, label=code_name, color=colors[code_name])
        axs[0,indexj].plot( dataEb_fit[code_name][j]['x_fit'], dataEb_fit[code_name][j]['Eb_fit'], '--', linewidth=1, color=colors[code_name], alpha = 0.6 )
        axs[0,indexj].fill_between(dataEb_fit[code_name][j]['x_fit'],dataEb_fit[code_name][j]['Eb_fit_lower_lim'],dataEb_fit[code_name][j]['Eb_fit_upper_lim'],color=colors[code_name],alpha=0.2)
    axs[0,indexj].axhline(y=-27,color='black',ls='--',label='CCSD(T)')


# Plot the top row binding energies
for indexj, j in enumerate(['Eb_LA','Eb_TM','Eb_DLA']):
    for i in ['CHAMP_Holland','CHAMP_Cornell_Paris','QMC=Chem','QMeCha','PyQMC','Amolqc','QWalk']:
        if j == 'Eb_LA' and i in ['QMC=Chem','Amolqc']: #'TurboRVB':
            continue
        elif j == 'Eb_TM' and i in ['QMC=Chem','QMeCha','Amolqc','CHAMP_Holland','CHAMP_Cornell_Paris']: #'TurboRVB':
            continue
        elif j == 'Eb_DLA' and i in ['QWalk','CHAMP_Holland','CHAMP_Cornell_Paris']: #'TurboRVB':
            continue
        code_name = i
        axs[1,indexj].errorbar(dataEb_fit[code_name][j]['x'], dataEb_fit[code_name][j]['Eb'],yerr = dataEb_fit[code_name][j]['Eb_err'], fmt='o', markersize=3,elinewidth=1, label=code_name, color=colors[code_name])
        axs[1,indexj].plot( dataEb_fit[code_name][j]['x_fit'], dataEb_fit[code_name][j]['Eb_fit'], '--', color=colors[code_name], linewidth=1, alpha = 0.6 )
        axs[1,indexj].fill_between(dataEb_fit[code_name][j]['x_fit'],dataEb_fit[code_name][j]['Eb_fit_lower_lim'],dataEb_fit[code_name][j]['Eb_fit_upper_lim'],color=colors[code_name],alpha=0.2)
    
    axs[1,indexj].axhline(y=-27,color='black',ls='--',label='CCSD(T)')



for i in code_final_list:
    axs[1,3].bar([1],[5],label=i, color=colors[i])

axs[1,3].axis('off')
axs[1,3].legend(fancybox=False, fontsize=7)


axs[0,0].set_title('LA')
axs[0,1].set_title('TM')
axs[0,2].set_title('DLA')
axs[0,3].set_title('DLTM')

axs[0,0].set_xlim([0.0,0.09])
axs[0,0].set_ylim([-70,-20])
axs[0,0].set_ylabel('Binding energy [meV]')
axs[1,0].set_ylabel('Binding energy [meV]')

axs[1,0].set_xticks([0.00,0.03,0.06,0.09])
axs[1,0].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,1].set_xticks([0.00,0.03,0.06,0.09])
axs[1,1].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,2].set_xticks([0.00,0.03,0.06,0.09])
axs[1,2].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

# axs[0,3].set_xticks([0.00,0.03,0.06,0.09])
# axs[0,3].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,1].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,2].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
# axs[1,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )


plt.savefig('Figures/Fig_S1.png')

# Figure S2 - Time-step dependence of total energies of individual codes

In [11]:
code_final_list

['CASINO',
 'TurboRVB',
 'TurboRVB_DMCv2',
 'QMCPACK_final',
 'CMQMC',
 'CHAMP_Holland',
 'CHAMP_Cornell_Paris',
 'QMC=Chem',
 'QMeCha',
 'PyQMC',
 'Amolqc',
 'QWalk']